# One-Hot Encoded Training Dataset — 4 Risk Groups

**Groups:**
- **Group 0 (Control):** Has diabetes + no hip fracture, OR hip fracture occurred >10 years after diabetes
- **Group 1 (Positive, <1yr):** Diabetes first, then hip fracture within 1 year
- **Group 2 (Positive, 1–5yr):** Diabetes first, then hip fracture 1–5 years after
- **Group 3 (Positive, 5–10yr):** Diabetes first, then hip fracture 5–10 years after

**Leakage prevention:** For positive groups (1, 2, 3), only codes with `first_intday < hip_first_day` are included.
Hip fracture codes are excluded from the feature set entirely.

**Source data:** `phe_aggregated_2024.pkl` — long-format, one row per (patient, phecode)
- 12,174,785 eligible patients (adult diabetics, enrolled >1 year)

In [1]:
import pandas as pd
import numpy as np
import pickle
import time
from scipy.sparse import csr_matrix, save_npz

## Step 1: Load data and drop NaN phecodes

In [2]:
DATA_PATH = '/Users/annagerasimenko/phe_2024_updated_scripts_and_models/phe_aggregated_2024.pkl'

print("Loading phe_aggregated_2024.pkl...")
t0 = time.time()
df = pd.read_pickle(DATA_PATH)
print(f"Loaded {len(df):,} rows in {time.time()-t0:.1f}s")
print(f"Columns: {df.columns.tolist()}")
df.head()

Loading phe_aggregated_2024.pkl...
Loaded 493,346,801 rows in 385.5s
Columns: ['enrolid', 'phecode', 'first_intday', 'last_intday', 'count', 'all_intdays']


,enrolid,phecode,first_intday,last_intday,count,all_intdays
0,2902,CV_401.1,125,6462,33,"[125, 140, 314, 329, 497, 514, 679, 496, 504, ..."
1,2902,DE_670,101,623,2,"[101, 623]"
2,2902,ID_089.2,101,101,1,[101]
3,2902,EM_239,329,6462,7,"[329, 679, 6000, 6016, 6364, 6429, 6462]"
4,2902,CA_138,623,623,1,[623]


In [3]:
n_before = len(df)
df = df.dropna(subset=['phecode'])
n_dropped = n_before - len(df)
print(f"Dropped {n_dropped:,} rows with NaN phecode ({n_dropped/n_before*100:.3f}%)")
print(f"Remaining: {len(df):,} rows, {df['enrolid'].nunique():,} unique patients")

Dropped 0 rows with NaN phecode (0.000%)
Remaining: 493,346,801 rows, 12,174,785 unique patients


## Step 2: Define phecode sets

PhecodeX codes used:
- **Diabetes:** `EM_202`, `EM_202.1`, `EM_202.2`, `BI_181` (from PHE2024_db_querying.ipynb)
- **Hip fracture:** `MS_745.11` (Fracture of femur), `MS_745.9` (Pathological fracture)
  — PhecodeX equivalents of the old phecode1.2 set {800.1, 800.2, 733.8, 743.22}

In [4]:
DIAB_PHE = {"EM_202", "EM_202.1", "EM_202.2", "BI_181"}
HIP_PHE  = {"MS_745.11", "MS_745.9"}

# Verify these codes exist in the data
found_diab = set(df['phecode'].unique()) & DIAB_PHE
found_hip  = set(df['phecode'].unique()) & HIP_PHE
print(f"Diabetes phecodes found in data:     {found_diab}")
print(f"Hip fracture phecodes found in data: {found_hip}")

Diabetes phecodes found in data:     {'BI_181', 'EM_202', 'EM_202.1', 'EM_202.2'}
Hip fracture phecodes found in data: {'MS_745.11', 'MS_745.9'}


## Step 3: Compute per-patient timelines

In [5]:
# Earliest diabetes diagnosis per patient
diab_first = (
    df[df['phecode'].isin(DIAB_PHE)]
    .groupby('enrolid')['first_intday']
    .min()
    .rename('diabetes_first_day')
    .reset_index()
)

# Earliest hip fracture diagnosis per patient
hip_first = (
    df[df['phecode'].isin(HIP_PHE)]
    .groupby('enrolid')['first_intday']
    .min()
    .rename('hip_first_day')
    .reset_index()
)

print(f"Patients with diabetes code:      {len(diab_first):,}")
print(f"Patients with hip fracture code:  {len(hip_first):,}")

Patients with diabetes code:      12,174,785
Patients with hip fracture code:  184,138


In [6]:
# Build per-patient timeline (left join — keep all diabetic patients)
timeline = diab_first.merge(hip_first, on='enrolid', how='left')

# Days from diabetes diagnosis to hip fracture (negative = hip BEFORE diabetes)
timeline['days_to_hip'] = timeline['hip_first_day'] - timeline['diabetes_first_day']

print(timeline.head(10))
print(f"\nPatients with hip fracture AFTER diabetes: {(timeline['days_to_hip'] > 0).sum():,}")
print(f"Patients with hip fracture BEFORE diabetes: {(timeline['days_to_hip'] <= 0).sum():,}")
print(f"Patients with NO hip fracture: {timeline['hip_first_day'].isna().sum():,}")

   enrolid  diabetes_first_day  hip_first_day  days_to_hip
0     2902                6040            NaN          NaN
1     3702                 188            NaN          NaN
2     3902                2953            NaN          NaN
3     4601                6064            NaN          NaN
4     6001                1950            NaN          NaN
5     6002                 184            NaN          NaN
6     6102                   8            NaN          NaN
7     8201                2303            NaN          NaN
8     8501                4698            NaN          NaN
9     9701                1641            NaN          NaN

Patients with hip fracture AFTER diabetes: 129,710
Patients with hip fracture BEFORE diabetes: 54,428
Patients with NO hip fracture: 11,990,647


## Step 4: Assign group labels (0–3)

In [7]:
Y1  = 365          # 1 year in days
Y5  = 5  * 365     # 5 years
Y10 = 10 * 365     # 10 years

def assign_group(days_to_hip):
    """Assign risk group based on days between diabetes and hip fracture.
    
    Group 0: no hip fracture, OR hip fracture before/same day as diabetes,
             OR hip fracture > 10 years after diabetes
    Group 1: hip fracture 0 < days <= 1 year after diabetes
    Group 2: hip fracture 1 year < days <= 5 years after diabetes
    Group 3: hip fracture 5 years < days <= 10 years after diabetes
    """
    if pd.isna(days_to_hip) or days_to_hip <= 0 or days_to_hip > Y10:
        return 0
    elif days_to_hip <= Y1:
        return 1
    elif days_to_hip <= Y5:
        return 2
    else:  # Y5 < days <= Y10
        return 3

timeline['group'] = timeline['days_to_hip'].apply(assign_group)

group_counts = timeline['group'].value_counts().sort_index()
print("Group distribution:")
for g, n in group_counts.items():
    labels = {
        0: "Control (no hip frac or >10yr)",
        1: "Hip frac within 1 year",
        2: "Hip frac 1–5 years",
        3: "Hip frac 5–10 years"
    }
    print(f"  Group {g} — {labels[g]}: {n:,}")

Group distribution:
  Group 0 — Control (no hip frac or >10yr): 12,055,663
  Group 1 — Hip frac within 1 year: 37,646
  Group 2 — Hip frac 1–5 years: 58,478
  Group 3 — Hip frac 5–10 years: 22,998


## Step 5: Compute feature cutoff day per patient

For positive groups (1, 2, 3): cutoff = `hip_first_day` (strict `<`, so day of fracture itself is excluded).
For control group (0): no cutoff — all codes are included.

> **Note on group-0 patients who had hip fracture >10yr after diabetes:**
> Their codes are NOT cut at any point (they are treated as pure controls).
> If you want to cut them at 10 years post-diabetes to create a fairer comparison window, 
> set `cutoff_day` = `diabetes_first_day + Y10` for those patients.

In [8]:
# For positive groups, cutoff = hip_first_day
# For controls (group 0), cutoff = NaN (no restriction)
timeline['cutoff_day'] = np.where(
    timeline['group'] > 0,
    timeline['hip_first_day'],
    np.nan
)

print("Cutoff day assigned:")
print(timeline[['enrolid', 'diabetes_first_day', 'hip_first_day', 'days_to_hip', 'group', 'cutoff_day']].head(10))

Cutoff day assigned:
   enrolid  diabetes_first_day  hip_first_day  days_to_hip  group  cutoff_day
0     2902                6040            NaN          NaN      0         NaN
1     3702                 188            NaN          NaN      0         NaN
2     3902                2953            NaN          NaN      0         NaN
3     4601                6064            NaN          NaN      0         NaN
4     6001                1950            NaN          NaN      0         NaN
5     6002                 184            NaN          NaN      0         NaN
6     6102                   8            NaN          NaN      0         NaN
7     8201                2303            NaN          NaN      0         NaN
8     8501                4698            NaN          NaN      0         NaN
9     9701                1641            NaN          NaN      0         NaN


## Step 6: Merge timeline info onto long-format data and apply cutoff

In [9]:
# Merge group label and cutoff day onto every (patient, phecode) row
t0 = time.time()
df = df.merge(
    timeline[['enrolid', 'group', 'cutoff_day', 'diabetes_first_day']],
    on='enrolid',
    how='inner'  # inner join: keeps only the 12M eligible diabetic patients
)
print(f"After merge: {len(df):,} rows  ({time.time()-t0:.1f}s)")

After merge: 493,346,801 rows  (105.8s)


In [10]:
# Remove hip fracture codes from all groups (prevent any leakage of the outcome itself)
n_before = len(df)
df = df[~df['phecode'].isin(HIP_PHE)]
print(f"Removed hip fracture phecode rows: {n_before - len(df):,}")

# Apply temporal cutoff for positive groups:
#   Keep a row if: (a) no cutoff applies (control), OR (b) code appeared before cutoff day
mask_no_cutoff   = df['cutoff_day'].isna()                       # group 0 controls
mask_before_hip  = df['first_intday'] < df['cutoff_day']          # code before hip fracture
df = df[mask_no_cutoff | mask_before_hip]

print(f"After temporal cutoff filter: {len(df):,} rows")
print(f"Unique patients remaining:    {df['enrolid'].nunique():,}")

Removed hip fracture phecode rows: 196,617
After temporal cutoff filter: 489,442,725 rows
Unique patients remaining:    12,174,785


## Step 7: Build one-hot encoded sparse matrix

In [11]:
# Ordered lists used to define row/column indices
all_patients  = df['enrolid'].unique()
all_phecodes  = df['phecode'].unique()

patient_to_idx = {pid: i for i, pid in enumerate(all_patients)}
phecode_to_idx = {code: i for i, code in enumerate(all_phecodes)}

print(f"Matrix dimensions: {len(all_patients):,} patients × {len(all_phecodes):,} phecodes")

Matrix dimensions: 12,174,785 patients × 3,304 phecodes


In [12]:
t0 = time.time()

row_indices = df['enrolid'].map(patient_to_idx).values
col_indices = df['phecode'].map(phecode_to_idx).values
data_vals   = np.ones(len(df), dtype=np.int8)

X = csr_matrix(
    (data_vals, (row_indices, col_indices)),
    shape=(len(all_patients), len(all_phecodes)),
    dtype=np.int8
)

print(f"Built sparse matrix in {time.time()-t0:.1f}s")
print(f"Shape:        {X.shape}")
print(f"Non-zeros:    {X.nnz:,}")
print(f"Density:      {X.nnz / (X.shape[0] * X.shape[1]) * 100:.4f}%")

Built sparse matrix in 75.9s
Shape:        (12174785, 3304)
Non-zeros:    489,442,725
Density:      1.2167%


In [13]:
# Build per-patient label array (group 0-3), aligned to all_patients order
patient_group_map = timeline.set_index('enrolid')['group']
y = np.array([patient_group_map[pid] for pid in all_patients], dtype=np.int8)

print("Final group distribution in y:")
group_labels = {
    0: "Control (no hip frac or >10yr)",
    1: "Hip frac within 1 year",
    2: "Hip frac 1–5 years",
    3: "Hip frac 5–10 years"
}
for g in range(4):
    n = (y == g).sum()
    print(f"  Group {g} — {group_labels[g]}: {n:,}")

Final group distribution in y:
  Group 0 — Control (no hip frac or >10yr): 12,055,663
  Group 1 — Hip frac within 1 year: 37,646
  Group 2 — Hip frac 1–5 years: 58,478
  Group 3 — Hip frac 5–10 years: 22,998


## Step 8: Save outputs

In [14]:
OUT_DIR = '/Users/annagerasimenko/phe_2024_updated_scripts_and_models/'

# Sparse feature matrix
save_npz(OUT_DIR + 'one_hot_features_4groups.npz', X)
print("Saved: one_hot_features_4groups.npz")

# Labels + metadata (enrolid order, phecode names, group labels)
np.savez_compressed(
    OUT_DIR + 'one_hot_labels_metadata_4groups.npz',
    y            = y,
    enrolid      = all_patients,
    feature_names= np.array(all_phecodes),
    group_labels = np.array([group_labels[g] for g in range(4)])
)
print("Saved: one_hot_labels_metadata_4groups.npz")
print("\nDone.")

Saved: one_hot_features_4groups.npz
Saved: one_hot_labels_metadata_4groups.npz

Done.


## Sanity check — reload and verify

In [15]:
from scipy.sparse import load_npz

X_check = load_npz(OUT_DIR + 'one_hot_features_4groups.npz')
meta    = np.load(OUT_DIR + 'one_hot_labels_metadata_4groups.npz', allow_pickle=True)

print(f"Feature matrix shape: {X_check.shape}")
print(f"Labels shape:         {meta['y'].shape}")
print(f"Feature names sample: {meta['feature_names'][:5]}")
print(f"Group labels:         {meta['group_labels']}")
print(f"Class counts:         {np.bincount(meta['y'])}")

Feature matrix shape: (12174785, 3304)
Labels shape:         (12174785,)
Feature names sample: ['CV_401.1' 'DE_670' 'ID_089.2' 'EM_239' 'CA_138']
Group labels:         ['Control (no hip frac or >10yr)' 'Hip frac within 1 year'
 'Hip frac 1–5 years' 'Hip frac 5–10 years']
Class counts:         [12055663    37646    58478    22998]
